In [ ]:
%gui asyncio

In [ ]:
import asyncio    
from ipylgbst import LegoBoostWidget,LedColor,Port, Sensor
from ipywidgets import Output
from IPython.display import display

# the robot's LED supports the following colors
colors = [
    LedColor.pink,
    LedColor.purple,
    LedColor.blue,
    LedColor.lightblue,
    LedColor.cyan,
    LedColor.green,
    LedColor.yellow,
    LedColor.orange,
    LedColor.red,
]

# Connect the device 
Note that connecting the device should take 5-10 sec

In [ ]:
boost = LegoBoostWidget()
connect_task = boost.connect()
boost

# Try simple sync and async tasks

In [ ]:
task1 = boost.set_led("pink")

In [ ]:
task2 = await boost.set_led_async("green")

In [ ]:
task3 = boost.motor_angle_multi(angle=200, power_a=50,power_b=-1.0*50)

In [ ]:
task4 = await boost.motor_angle_multi_async(angle=400, power_a=50,power_b=-1.0*50)

# Swipe all LED colors

In [ ]:
for color in colors:
    print(f"set color to {color.value}")
    await boost.set_led_async(color)
    await asyncio.sleep(3.0)

# Drive a polygon like path

In [ ]:
speed = 50
d_forward  = 200
d_rotate = 300
for i in range(4):
    await boost.set_led_async("red")
    await boost.motor_angle_multi_async(angle=d_forward, power_a=speed,power_b=speed)
    await boost.set_led_async("green")
    await boost.motor_angle_multi_async(angle=d_rotate, power_a=speed,power_b=-1.0*speed)

# Buttons (sync but non blocking API)

the non-blocking sync API is the best for a "remote-controlled-rc-car" experience

In [ ]:
import ipywidgets as widgets
from IPython.display import display

button_up = widgets.Button(description='Up')

button_down = widgets.Button(description='Down')
button_left = widgets.Button(description='Left')
button_right = widgets.Button(description='Right')
button_stop  = widgets.Button(description='Stop')

box_ud = widgets.VBox([button_up,button_stop,button_down])
box = widgets.HBox([button_left,box_ud,button_right])

def on_up(_):
    # run sync but NON blocking
    boost.motor_time_multi(10, 50, 50)
    
def on_down(_):
    # run sync but NON blocking
    boost.motor_time_multi(10, -50, -50)
    
def on_left(_):
    # run sync but NON blocking
    boost.motor_time_multi(10, -50, 50)
    
def on_right(_):
    # run sync but NON blocking
    boost.motor_time_multi(10, 50, -50)
    
def on_stop(_):
    # run sync but NON blocking
    boost.motor_time_multi(0, 0, 0)

    
button_up.on_click(on_up) 
button_down.on_click(on_down) 
button_left.on_click(on_left) 
button_right.on_click(on_right) 
button_stop.on_click(on_stop)

display(box)

# Sensors

In [ ]:
has_bqplot = True

try:
    import bqplot
except ImportError:
    has_bqplot = False

if has_bqplot:
    from bqplot import pyplot as plt
    import numpy
    import asyncio

    duration = 10.0
    dt = 0.10
    clip_value = 200

    sensor_values = []
    time_points = []

    # Create the figure and the line
    plt.figure()
    line = plt.plot([], [])

    plt.xlim(0, duration)
    plt.ylim(0, clip_value)
    plt.xlabel("time")
    plt.ylabel("distance sensor")
    plt.show()

    async def poll_distance_sensor():
        t = 0.0

        while t < duration:
            d = await boost.get_distance_async()
            d = numpy.clip(d, 0, clip_value)

            time_points.append(t)
            sensor_values.append(d)

            # Update the existing line
            line.x = time_points
            line.y = sensor_values

            await asyncio.sleep(dt)
            t += dt

    await poll_distance_sensor()


# Concurrent programs

In [ ]:
import asyncio
import ipywidgets as widgets
from IPython.display import display

speed = 26 
d_forward = 200 
d_rotate = 300

async def set_leds():
    for i, color in enumerate(colors):
        print(f"LED {i}: change color to {color.value}")

        await boost.set_led_async(color)
        await asyncio.sleep(0.5)



async def drive():
    for i in range(4):
        print(f"Move forward {i}")
        await boost.motor_angle_multi_async(
            angle=d_forward,
            power_a=speed,
            power_b=speed
        )
        print(f"Rotate {i}")
        await boost.motor_angle_multi_async(
            angle=d_rotate,
            power_a=speed,
            power_b=-speed
        )

async def run():
    await asyncio.gather(
        set_leds(),
        drive()
    )


await run()